# Exploração dos dados brutos — Banco de Preços em Saúde (BPS)

**Sprint 1 — Entendimento do problema e dos dados**

## Por que este notebook existe

Antes de escrever qualquer linha de código de limpeza, eu precisava responder a uma pergunta simples: **os 7 arquivos (2020 a 2026) têm exatamente a mesma estrutura?**

Isso parece óbvio, mas na prática quase nunca é verdade em bases públicas com longo histórico. Ao longo de 7 anos, é comum que:
- O sistema que gera os dados seja atualizado, e colunas mudem de nome ou sejam adicionadas/removidas
- Arquivos de anos diferentes sejam exportados por pessoas ou processos diferentes, e o encoding de caracteres (a forma como acentos são salvos) mude
- O separador de colunas varie

Se eu simplesmente juntasse os 7 CSVs sem checar isso primeiro, o risco era gerar um arquivo final com colunas duplicadas, dados corrompidos (tipo "PreÃ§o" no lugar de "Preço") ou, pior, um erro silencioso onde tudo parece ter funcionado mas os dados estão errados.

## Localizando a pasta de dados

O notebook pode ser executado a partir de pastas de trabalho diferentes dependendo do ambiente (localmente isso variou até durante o desenvolvimento). Em vez de fixar um caminho relativo frágil, a função abaixo sobe pelos diretórios a partir da localização atual até encontrar uma pasta que contenha `data/raw` — essa é considerada a raiz do projeto. Isso garante que o notebook funcione de forma reprodutível em qualquer máquina.

In [ ]:
from pathlib import Path


def encontrar_raiz_projeto(inicio: Path, marcador=("data", "raw")) -> Path:
    """Sobe pelos diretórios a partir de 'inicio' até achar uma pasta
    que contenha data/raw dentro dela — essa é a raiz do projeto."""
    atual = inicio.resolve()
    candidatos = [atual] + list(atual.parents)
    for pasta in candidatos:
        if (pasta.joinpath(*marcador)).exists():
            return pasta
    raise FileNotFoundError(
        f"Não encontrei uma pasta 'data/raw' subindo a partir de {inicio}"
    )


RAIZ_PROJETO = encontrar_raiz_projeto(Path.cwd())
PASTA_DADOS = RAIZ_PROJETO / "data" / "raw"

print("Raiz do projeto encontrada:", RAIZ_PROJETO)
print("Pasta de dados:", PASTA_DADOS)
print("Existe?", PASTA_DADOS.exists())

Raiz do projeto encontrada: C:\Users\waldinei.rosa\OneDrive\Documentos\SCTEC\Modulo_2\Mini_Projeto\projeto_bps_waldinei
Pasta de dados: C:\Users\waldinei.rosa\OneDrive\Documentos\SCTEC\Modulo_2\Mini_Projeto\projeto_bps_waldinei\data\raw
Existe? True


## Inspecionando o arquivo de 2020 byte a byte

Antes de comparar os 7 anos entre si, inspecionei manualmente o arquivo de 2020 sem usar pandas, lendo os bytes crus do cabeçalho e da primeira linha de dados. Isso confirmou dois pontos importantes:

- O separador de colunas é `;` (ponto e vírgula), e não `,` — comum em arquivos brasileiros, já que a vírgula é usada como separador decimal.
- O arquivo está salvo em **UTF-8**: decodificar a primeira linha de dados nesse encoding preserva corretamente acentos como em "SÓDICA" e "dentário". Uma tentativa de decodificação em Latin-1 (ISO-8859-1), por comparação, corrompe esses mesmos caracteres (ex.: "SÃDICA", "dentÃ¡rio") — confirmando que UTF-8 é o encoding correto.

Também notei que os nomes de coluna no CSV são técnicos (`no_instituicao`, `vl_preco_unitario`...) e não batem literalmente com os nomes "amigáveis" do dicionário de dados oficial do BPS (`Nome Instituição`, `Preço Unitário`...). Esse de-para será documentado na seção de descrição de colunas do README final.

In [9]:
caminho_2020 = PASTA_DADOS / "2020.csv"

with open(caminho_2020, "rb") as f:
    cabecalho = f.readline()
    primeira_linha = f.readline()

print("--- CABEÇALHO (bruto) ---")
print(cabecalho)

print("\n--- PRIMEIRA LINHA DE DADOS (decodificada em UTF-8) ---")
print(primeira_linha.decode("utf-8"))

print("\n--- MESMA LINHA, TENTATIVA EM LATIN-1 (para comparação) ---")
print(primeira_linha.decode("latin-1"))

--- CABEÇALHO (bruto) ---
b'ano_compra;"cnpj_instituicao";"sg_uf";"ds_esfera";dt_compra;dt_insercao;validade_compra;"co_catmat";"ds_item";"co_pdm";"co_grupo";"no_grupo";"co_classe";"no_classe";"fg_generico";"tp_compra";"sg_unidade_medida";"cnpj_fornecedor";"no_fornecedor";"cnpj_fabricante";"no_fabricante";qt_medicamento;"ds_observacao";"no_instituicao";"no_municipio";"un_medida_capacidade";"no_pdm";"nu_processo_compra";"nu_ata";"un_fornecimento";"registro_anvisa";"modalidade";vl_capacidade;vl_preco_unitario;vl_preco_total;co_seq_bps\r\n'

--- PRIMEIRA LINHA DE DADOS (decodificada em UTF-8) ---
"2020";"21467008000132";"RO";"MUNICIPAL";"21/10/2020";"11/09/2025";"12";"267203";"DIPIRONA SÓDICA, DOSAGEM:500 MG";"17708";"65";"Equipamentos e artigos para uso médico, dentário e veterinario";"6505";"DROGAS E MEDICAMENTOS";;"ADMINISTRATIVA";;"16970999000131";"DMC DISTRIBUIDORAS, COMERCIO D MEDICAMENTOS LTDA";"33408105000133";"GREENPHARMA QUIMICA E FARMACEUTICA EM RECUPERACAO JUDICIAL LTDA";"5000

## Comparando os 7 anos de uma vez

Com a estrutura do arquivo de 2020 entendida, repito agora o mesmo teste — cabeçalho e encoding — para os 7 anos de uma vez, de forma sistemática, para descobrir se algum ano foge do padrão.

In [10]:
anos = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
cabecalhos = {}

for ano in anos:
    caminho = PASTA_DADOS / f"{ano}.csv"
    with open(caminho, "rb") as f:
        cabecalho = f.readline()
        primeira_linha = f.readline()

    cabecalhos[ano] = cabecalho.decode("utf-8", errors="replace")

    print(f"===== {ano} =====")
    print("Nº de colunas:", len(cabecalhos[ano].split(";")))
    try:
        primeira_linha.decode("utf-8")
        print("Encoding: UTF-8 OK")
    except UnicodeDecodeError:
        print("Encoding: NÃO é UTF-8 válido")
    print()

print("=== COMPARAÇÃO COM 2020 ===")
for ano in anos:
    igual = cabecalhos[ano] == cabecalhos[2020]
    print(f"{ano}: {'idêntico a 2020' if igual else 'DIFERENTE de 2020'}")

===== 2020 =====
Nº de colunas: 36
Encoding: UTF-8 OK

===== 2021 =====
Nº de colunas: 36
Encoding: UTF-8 OK

===== 2022 =====
Nº de colunas: 36
Encoding: UTF-8 OK

===== 2023 =====
Nº de colunas: 36
Encoding: UTF-8 OK

===== 2024 =====
Nº de colunas: 36
Encoding: UTF-8 OK

===== 2025 =====
Nº de colunas: 36
Encoding: UTF-8 OK

===== 2026 =====
Nº de colunas: 36
Encoding: UTF-8 OK

=== COMPARAÇÃO COM 2020 ===
2020: idêntico a 2020
2021: idêntico a 2020
2022: idêntico a 2020
2023: idêntico a 2020
2024: idêntico a 2020
2025: idêntico a 2020
2026: idêntico a 2020


## Conclusão

Os 7 anos (2020–2026) apresentam estrutura idêntica: mesmo número de colunas (36), mesmo cabeçalho e encoding UTF-8 válido em todos. Isso significa que a concatenação (Sprint 2) pode ser feita por simples empilhamento (`pd.concat`), sem necessidade de remapear colunas entre anos.

O detalhamento completo dessa conclusão está documentado em `docs/discrepancias_anos.md`.